# Movie data preprocessing
Load and combine train/test rows before applying every feature transformation to `data`. Category vocabularies and frequency rankings use the combined data. Test revenue remains missing. Original columns and row order are preserved.


In [1]:
import ast
import re
from pathlib import Path

import numpy as np
import pandas as pd

# Support running from either the repository root or the data directory.
path = Path.cwd() / "data" / "tmdb-box-office-prediction"
if not path.is_dir():
    path = Path.cwd() / "tmdb-box-office-prediction"

data_train = pd.read_csv(path / "train.csv")
data_test = pd.read_csv(path / "test.csv")
train_ids = data_train["id"].copy()
test_ids = data_test["id"].copy()
data = pd.concat(
    [data_train.assign(train_test="train"), data_test.assign(train_test="test")],
    ignore_index=True,
)
original_columns = data.columns.tolist()
print(f"Combined rows: {len(data):,}")


Combined rows: 7,398


## Shared parsing and encoding helpers
Parse serialized lists once. Missing or malformed values become empty lists. Company and country counts use unique names. Their `others` flag retains the existing meaning: no selected category is present, including empty lists.


In [2]:
def extract_records(value):
    if isinstance(value, str):
        if not value.strip():
            return []
        try:
            value = ast.literal_eval(value)
        except (ValueError, SyntaxError):
            return []
    if not isinstance(value, (list, tuple)):
        return []
    return [record for record in value if isinstance(record, dict)]


def extract_names(records):
    return sorted({record["name"] for record in records if record.get("name")})


def feature_column(prefix, name):
    normalized = re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")
    return f"{prefix}_{normalized}"


def encode_names(name_lists, selected_names, prefix, include_others=False):
    columns = [feature_column(prefix, name) for name in selected_names]
    if include_others:
        columns.append(f"{prefix}_others")
    if len(columns) != len(set(columns)):
        raise ValueError(f"Duplicate normalized feature names for {prefix}")
    features = pd.DataFrame(
        {
            feature_column(prefix, name): name_lists.map(
                lambda names, selected=name: int(selected in names)
            )
            for name in selected_names
        },
        index=name_lists.index,
    )
    if include_others:
        selected = set(selected_names)
        features[f"{prefix}_others"] = name_lists.map(
            lambda names: int(not selected.intersection(names))
        )
    return features


def member_value(records, key, match, field, default):
    record = next((record for record in records if record.get(key) == match), {})
    value = record.get(field)
    return default if value is None else value


parsed = {
    column: data[column].map(extract_records)
    for column in ["genres", "production_companies", "production_countries", "cast", "crew"]
}


## Presence flags and logarithms
`log_revenue` is a transformed target, not an input feature. It stays missing for test rows.


In [3]:
for source in ["belongs_to_collection", "homepage"]:
    values = data[source].astype("string")
    data[f"has_{source}"] = (values.notna() & values.str.strip().ne("")).astype(int)

for column in ["budget", "revenue"]:
    values = pd.to_numeric(data[column], errors="coerce")
    data[f"log_{column}"] = np.log1p(values)


## Categorical features
Encode all genres, the top 15 original languages, the top 40 production companies, and the top 20 production countries using the combined rows.


In [4]:
genre_lists = parsed["genres"].map(extract_names)
all_genres = sorted({name for names in genre_lists for name in names})
genre_features = encode_names(genre_lists, all_genres, "genre")
genre_columns = genre_features.columns.tolist()
data[genre_columns] = genre_features

language_values = data["original_language"].astype("string").str.strip().replace("", pd.NA)
top_15_languages = language_values.value_counts().head(15).index.tolist()
language_features = pd.DataFrame(
    {
        f"original_language_{language}": language_values.eq(language).fillna(False).astype(int)
        for language in top_15_languages
    },
    index=data.index,
)
language_features["original_language_others"] = (~language_values.isin(top_15_languages)).astype(int)
language_columns = language_features.columns.tolist()
data[language_columns] = language_features

production_company_lists = parsed["production_companies"].map(extract_names)
top_40_production_companies = production_company_lists.explode().value_counts().head(40).index.tolist()
company_features = encode_names(
    production_company_lists, top_40_production_companies, "production_company", include_others=True
)
production_company_columns = company_features.columns.tolist()
data[production_company_columns] = company_features

production_country_lists = parsed["production_countries"].map(extract_names)
top_20_production_countries = production_country_lists.explode().value_counts().head(20).index.tolist()
country_features = encode_names(
    production_country_lists, top_20_production_countries, "production_country", include_others=True
)
production_country_columns = country_features.columns.tolist()
data[production_country_columns] = country_features
# Consolidate blocks before adding the remaining features.
data = data.copy()


C:\Users\luyer\AppData\Local\Temp\ipykernel_27212\3414829356.py:34: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data[production_country_columns] = country_features


## Production counts and release dates
Dates retain the existing `%m/%d/%y` parsing convention; unparseable dates remain missing.


In [5]:
data["number_of_production_companies"] = production_company_lists.map(len)
data["number_of_production_countries"] = production_country_lists.map(len)
release_dates = pd.to_datetime(data["release_date"], format="%m/%d/%y", errors="coerce")
data["release_year"] = release_dates.dt.year
data["release_month"] = release_dates.dt.month
# Monday = 0, Sunday = 6; missing dates remain missing.
data["release_day_of_week"] = release_dates.dt.dayofweek


## Cast and crew
Use billing `order` 0 and 1 for the first two cast members. Select the first director and producer listed. Missing names use `Unknown`; missing genders use -1.


In [6]:
cast_feature_columns = []
for position, cast_order in [(1, 0), (2, 1)]:
    for field, default in [("name", "Unknown"), ("gender", -1)]:
        column = f"cast_{position}_{field}"
        data[column] = parsed["cast"].map(
            lambda records, order=cast_order, field=field, default=default:
                member_value(records, "order", order, field, default)
        )
        if field == "gender":
            data[column] = data[column].astype(int)
        cast_feature_columns.append(column)

crew_feature_columns = []
for role in ["Director", "Producer"]:
    for field, default in [("name", "Unknown"), ("gender", -1)]:
        column = f"{role.lower()}_{field}"
        data[column] = parsed["crew"].map(
            lambda records, role=role, field=field, default=default:
                member_value(records, "job", role, field, default)
        )
        if field == "gender":
            data[column] = data[column].astype(int)
        crew_feature_columns.append(column)


## Frequent cast, directors, producers, and keywords
Use all combined train/test movies to select the top 50 names per people group and top 100 keywords by movie frequency. Multiple indicators can be active per movie. Each others flag includes any unselected name or keyword, as well as empty lists.


In [7]:
# Count each name once per movie, including all cast and matching crew members.
category_name_lists = {
    "cast": parsed["cast"].map(extract_names),
    "director": parsed["crew"].map(
        lambda records: extract_names([r for r in records if r.get("job") == "Director"])
    ),
    "producer": parsed["crew"].map(
        lambda records: extract_names([r for r in records if r.get("job") == "Producer"])
    ),
    "keyword": data["Keywords"].map(extract_records).map(extract_names),
}
category_limits = {"cast": 50, "director": 50, "producer": 50, "keyword": 100}
top_category_counts = {}
top_category_names = {}
category_feature_blocks = []

for category, names in category_name_lists.items():
    # Alphabetical tie-breaking makes the selected vocabulary reproducible.
    counts = names.explode().dropna().value_counts()
    counts = counts.sort_index().sort_values(ascending=False, kind="stable")
    top_category_counts[category] = counts.head(category_limits[category])
    selected = top_category_counts[category].index.tolist()
    top_category_names[category] = selected
    features = encode_names(names, selected, category, include_others=True)
    selected_set = set(selected)
    # A movie may contain both a selected name and an unselected name.
    # Empty/missing lists also belong to the fallback category.
    features[f"{category}_others"] = names.map(
        lambda values, selected_set=selected_set:
            int(not values or bool(set(values) - selected_set))
    )
    category_feature_blocks.append(features.astype("int8"))

people_keyword_features = pd.concat(category_feature_blocks, axis=1)
people_keyword_columns = people_keyword_features.columns.tolist()
data = pd.concat(
    [data.drop(columns=people_keyword_columns, errors="ignore"), people_keyword_features],
    axis=1,
).copy()

assert len(people_keyword_columns) == sum(category_limits.values()) + len(category_limits)
assert data[people_keyword_columns].isin([0, 1]).all().all()
for category, names in category_name_lists.items():
    selected = set(top_category_names[category])
    expected_others = names.map(lambda values: int(not values or bool(set(values) - selected)))
    assert data[f"{category}_others"].eq(expected_others).all()
    for name in top_category_names[category]:
        assert data[feature_column(category, name)].eq(names.map(lambda values: int(name in values))).all()

print(f"Added {len(people_keyword_columns)} cast, director, producer, and keyword indicators.")
# Inspect the selected categories and their movie counts.
pd.concat(
    [counts.rename_axis("name").reset_index(name="movie_count").assign(category=category)
     for category, counts in top_category_counts.items()],
    ignore_index=True,
)[["category", "name", "movie_count"]]


Added 254 cast, director, producer, and keyword indicators.


,category,name,movie_count
0,cast,Samuel L. Jackson,80
1,cast,Robert De Niro,71
2,cast,Bruce Willis,62
3,cast,Morgan Freeman,61
4,cast,Liam Neeson,57
...,...,...,...
245,keyword,alcohol,55
246,keyword,fbi,55
247,keyword,romance,55
248,keyword,scientist,55


## Validate and split the transformed data
`data` is the complete transformed dataframe. Recreate train/test views only after all transformations so both have the same feature columns. Exclude `revenue`, `log_revenue`, and the split marker from model inputs.


In [8]:
feature_columns = [column for column in data.columns if column not in original_columns]
assert data.columns.is_unique
assert len(data) == len(train_ids) + len(test_ids)
assert data.loc[data["train_test"].eq("test"), ["revenue", "log_revenue"]].isna().all().all()
assert data[language_columns].sum(axis=1).eq(1).all()

binary_columns = genre_columns + language_columns + production_company_columns + production_country_columns + people_keyword_columns
assert data[binary_columns].isin([0, 1]).all().all()

data_train = data.loc[data["train_test"].eq("train")].copy().reset_index(drop=True)
data_test = data.loc[data["train_test"].eq("test")].copy().reset_index(drop=True)
assert data_train["id"].equals(train_ids)
assert data_test["id"].equals(test_ids)
assert data_train.columns.equals(data_test.columns)

print(f"Created {len(feature_columns)} features on {len(data):,} combined rows.")
print(f"Train: {data_train.shape}; test: {data_test.shape}")
data[["id", "train_test"] + feature_columns].head()


Created 369 features on 7,398 combined rows.
Train: (3000, 393); test: (4398, 393)


,id,train_test,has_belongs_to_collection,has_homepage,log_budget,log_revenue,genre_action,genre_adventure,genre_animation,genre_comedy,...,keyword_shootout,keyword_assassin,keyword_extramarital_affair,keyword_psychopath,keyword_alcohol,keyword_fbi,keyword_romance,keyword_scientist,keyword_adultery,keyword_others
0,1,train,1,0,16.454568,16.326300,0,0,0,1,...,0,0,0,0,0,0,0,0,0,1
1,2,train,1,0,17.504390,18.370959,0,0,0,1,...,0,0,0,0,0,0,0,0,0,1
2,3,train,0,1,15.009433,16.387512,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
3,4,train,0,1,13.997833,16.588099,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
4,5,train,0,0,0.000000,15.182615,1,0,0,0,...,0,0,0,0,0,0,0,0,0,1


In [10]:
#save both transformed train and test data to csv
data_train.to_csv(path / "train_transformed.csv", index=False)
data_test.to_csv(path / "test_transformed.csv", index=False)